# 零售订单经营指标分析项目

本 notebook 展示项目的核心分析流程：读取数据、清洗订单、计算经营指标、客户 RFM 分层，并输出可导入 Power BI 的数据表。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("..")
raw_path = BASE / "data" / "processed" / "online_retail_raw.csv"
df = pd.read_csv(raw_path, parse_dates=["InvoiceDate"])
df.head()

## 1. 数据清洗

删除取消订单、非正数量、非正价格，并新增销售额和月份字段。

In [ ]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["is_cancelled"] = df["InvoiceNo"].str.upper().str.startswith("C")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)

sales = df[(~df["is_cancelled"]) & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
sales.shape

## 2. 月度经营指标

In [ ]:
monthly = sales.groupby("InvoiceMonth").agg(
    revenue=("Revenue", "sum"),
    orders=("InvoiceNo", "nunique"),
    customers=("CustomerID", "nunique"),
    units_sold=("Quantity", "sum"),
).reset_index()
monthly["avg_order_value"] = monthly["revenue"] / monthly["orders"]
monthly.head()

## 3. 商品和地区表现

In [ ]:
country = sales.groupby("Country").agg(revenue=("Revenue", "sum"), orders=("InvoiceNo", "nunique")).reset_index().sort_values("revenue", ascending=False)
product = sales.groupby(["StockCode", "Description"]).agg(revenue=("Revenue", "sum"), units_sold=("Quantity", "sum")).reset_index().sort_values("revenue", ascending=False)
country.head(), product.head()

## 4. 基础客户 RFM 分层

In [ ]:
customer_sales = sales.dropna(subset=["CustomerID"]).copy()
snapshot = customer_sales["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = customer_sales.groupby("CustomerID").agg(
    recency_days=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    frequency=("InvoiceNo", "nunique"),
    monetary=("Revenue", "sum"),
).reset_index()
rfm.head()

## 5. 输出结果

完整脚本见 `src/run_analysis.py`，可直接生成 `powerbi_data/` 和 `output/` 文件夹。